# TRELLIS Image-to-3D — Backend Colab (GPU L4)
Chọn **Runtime -> Change runtime type -> L4 GPU**, sau đó chạy lần lượt **3 cell** bên dưới.


In [ ]:
# Cell 1: Kiểm tra GPU, clone repo của team & cài Miniconda
!nvidia-smi
!rm -rf /content/TRELLIS
!git clone https://github.com/truong-newbie/image-to-3d-web.git /content/TRELLIS
!rm -rf /content/miniconda3
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /content/miniconda3
!rm /tmp/miniconda.sh


In [ ]:
%%bash
# Cell 2: Cài đặt môi trường trellis chuẩn cho Colab L4 (Ubuntu 24.04)
set -e
cd /content/TRELLIS
rm -rf /tmp/extensions

source /content/miniconda3/etc/profile.d/conda.sh
export CONDA_ALWAYS_YES="true"

conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

conda remove -n trellis --all -y || true
conda create -n trellis python=3.10 -y
conda activate trellis

conda install -y -c nvidia/label/cuda-12.8.0 \
  "cuda-nvcc=12.8*" "cuda-cudart-dev=12.8*" "cuda-cccl=12.8*" \
  libcublas-dev libcusparse-dev libcusolver-dev libcurand-dev libcufft-dev

export CUDA_HOME=/content/miniconda3/envs/trellis
export PATH=$CUDA_HOME/bin:$PATH
export CPATH=$CUDA_HOME/include:$CUDA_HOME/targets/x86_64-linux/include:${CPATH:-}
export LIBRARY_PATH=$CUDA_HOME/lib:$CUDA_HOME/targets/x86_64-linux/lib:${LIBRARY_PATH:-}
export TORCH_CUDA_ARCH_LIST="8.9"
export NVCC_APPEND_FLAGS="-allow-unsupported-compiler"

pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
pip install xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
pip install kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html

. ./setup.sh --basic
pip install "transformers==4.49.0" spconv-cu120 fastapi uvicorn python-multipart

mkdir -p /tmp/extensions
git clone https://github.com/NVlabs/nvdiffrast.git /tmp/extensions/nvdiffrast
pip install --no-build-isolation /tmp/extensions/nvdiffrast

git clone --recurse-submodules https://github.com/JeffreyXiang/diffoctreerast.git /tmp/extensions/diffoctreerast
pip install --no-build-isolation /tmp/extensions/diffoctreerast

git clone https://github.com/autonomousvision/mip-splatting.git /tmp/extensions/mip-splatting
pip install --no-build-isolation /tmp/extensions/mip-splatting/submodules/diff-gaussian-rasterization/

echo "Cài đặt hoàn tất!"


In [ ]:
# Cell 3: Cập nhật code mới nhất từ GitHub, khởi động Backend API & mở link Cloudflare Tunnel
import subprocess, time, urllib.request, json, re, os

subprocess.run("git -C /content/TRELLIS pull --ff-only || true", shell=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True
    )

subprocess.run("pkill -f 'uvicorn server:app' || true", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)

env = os.environ.copy()
env["ATTN_BACKEND"] = "xformers"
env["SPCONV_ALGO"] = "native"
api_log = open("/tmp/api.log", "w")
api_proc = subprocess.Popen(
    ["/content/miniconda3/envs/trellis/bin/uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/TRELLIS/web_app/backend",
    env=env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

print("Đang khởi động Backend & load model TRELLIS...", flush=True)
while True:
    if api_proc.poll() is not None:
        print("Lỗi khởi động API! Log:", flush=True)
        print(open("/tmp/api.log").read())
        raise RuntimeError("API crashed")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            data = json.loads(r.read().decode())
            if data.get("model_loaded"):
                print(f"Backend sẵn sàng trên GPU: {data.get('gpu')}", flush=True)
                break
    except Exception:
        pass
    time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in tunnel_proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n>>> LINK API CỦA BẠN:", m.group(0), flush=True)
        print(">>> Giữ cell này tiếp tục chạy trong lúc dùng Web UI nhé!\n", flush=True)
